# Irodori-TTS VoiceDesign Lab

Irodori-TTS の VoiceDesign checkpoint を使って、同じ自己紹介文を複数の話し方で生成するノートブックです。

最初のモデルロードと Hugging Face からのダウンロードは時間がかかります。2 回目以降はキャッシュされます。

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
from IPython.display import Audio, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT_DIR = ROOT / "outputs" / "voice_design_lab"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv が見つかりません。ターミナルで `which uv` を確認してください。")

def run_in_project(args, *, timeout=None):
    command = [UV, *args]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=False,
        timeout=timeout,
    )

print("root:", ROOT)
print("notebook python:", sys.executable)
print("uv:", UV)
print("outputs:", OUT_DIR)
run_in_project(["run", "python", "-c", "import sys, torch; from irodori_tts.inference_runtime import default_runtime_device; print('project python:', sys.executable); print('torch:', torch.__version__); print('device:', default_runtime_device())"])


In [ ]:
# このノートブックは kernel 環境に依存しないよう、推論は常に `uv run python infer.py` で実行します。
# 初回実行時は Hugging Face から model.safetensors と codec weights をダウンロードします。
HF_CHECKPOINT = "Aratako/Irodori-TTS-500M-v2-VoiceDesign"
print("checkpoint:", HF_CHECKPOINT)
print("inference entrypoint:", ROOT / "infer.py")


In [ ]:
TEXT = """
はじめまして。私は彩人です。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。
""".strip()

STYLE_PRESETS = {
    "confident_clear": "明るく自信に満ちた若い男性の声。背筋を伸ばして、ハキハキと、語尾まで明瞭に、少し速めのテンポで堂々と自己紹介してください。",
    "confident_calm": "落ち着いた自信のある男性の声。低すぎない自然な声で、聞き手に安心感を与えるように、ゆっくり丁寧に、しかし迷いなく話してください。",
    "nervous_mumbling": "自信がなさそうな若い男性の声。小さめの声で、少し息が漏れるように、ところどころ迷いながら、ボソボソと控えめに自己紹介してください。",
    "shy_soft": "内気でやわらかい男性の声。距離感は近く、声量は控えめで、少し照れながら、優しく自然に話してください。",
}

TEXT


In [ ]:
def synthesize_style(
    name: str,
    caption: str,
    *,
    seed: int = 42,
    num_steps: int = 24,
    cfg_scale_text: float = 3.0,
    cfg_scale_caption: float = 3.5,
) -> Path:
    out_path = OUT_DIR / f"{name}_seed{seed}_steps{num_steps}.wav"
    args = [
        "run",
        "python",
        "infer.py",
        "--hf-checkpoint",
        HF_CHECKPOINT,
        "--text",
        TEXT,
        "--caption",
        caption,
        "--no-ref",
        "--num-steps",
        str(num_steps),
        "--seed",
        str(seed),
        "--cfg-scale-text",
        str(cfg_scale_text),
        "--cfg-scale-caption",
        str(cfg_scale_caption),
        "--output-wav",
        str(out_path),
    ]
    run_in_project(args, timeout=900)
    print("saved:", out_path)
    return out_path


In [ ]:
# まずは狙いの 2 種類だけ生成します。
paths = []
for name in ["confident_clear", "nervous_mumbling"]:
    print("\n===", name, "===")
    paths.append(synthesize_style(name, STYLE_PRESETS[name], seed=20260505, num_steps=24))

for path in paths:
    print(path.name)
    display(Audio(filename=str(path)))


In [ ]:
# 追加比較。必要なものだけコメントアウトを外してください。
# extra_paths = []
# for name, caption in STYLE_PRESETS.items():
#     print("\n===", name, "===")
#     extra_paths.append(synthesize_style(name, caption, seed=20260505, num_steps=32))
#
# for path in extra_paths:
#     print(path.name)
#     display(Audio(filename=str(path)))


## 調整のコツ

- 話し方が弱いときは `cfg_scale_caption` を `4.0` から `5.0` くらいに上げる。
- 音が不安定なときは `num_steps` を `32` から `40` に上げる。
- 同じ caption で別の声にしたいときは `seed` を変える。
- まず素早く試すときは `num_steps=12`、本番候補は `num_steps=32` 以上がおすすめ。